[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/nb/warmup/NB_Warmup_07_NCF_Recap.ipynb)

# NB Warm-up 07 — NCF with a user tower: implement → fit → GridSearchCV

**STAT3009 · Recommender Systems**  
**Suggested time:** 18–20 minutes

> **Workflow warm-up:** keep the MF biases, replace the fixed user factor with a small user tower, and choose the network size with cross-validation.

**Route:** model and shapes (3 min) → data (2 min) → implement (5 min) → fit (3 min) → `GridSearchCV` (6 min) → exit ticket (2 min). We reuse the 16 ratings from Warm-up 04; no download is needed.

## 1. The model in one minute

$$
\widehat r_{ui}=\mu+a_u+b_i+f_u(p_u)^\mathsf{T}q_i.
$$

| Question | Answer |
|---|---|
| **Model** | baseline $\mu+a_u+b_i$ plus a **user-tower** interaction |
| **Hyperparameters** | embedding size $K$, tower width `hidden`, learning rate, epochs |
| **Fitting criterion** | squared error on the observed training pairs |
| **Learned parameters** | $\mu$, biases $a,b$, embeddings $P,Q$, and the tower weights |

Here $p_u$ and $q_i$ are both $K$-dimensional embeddings. The user tower $f_u$ maps $p_u$ to another $K$-dimensional vector; we then dot it with $q_i$.

> **Notation.** `f_u` is the user-side encoder. In code, one shared `self.user_tower` is applied to every user embedding. The subscript `u` identifies the input, not a separate network per user.

Full NCF concatenates $[p_u,q_i]$ and learns an MLP on the pair. This warm-up isolates one tower: transform $p_u$, then dot with $q_i$. If the tower is the identity, $f_u(p_u)=p_u$, and the model is exactly biased MF. The tower is the only change.

### Shape contract

| tensor | shape |
|---|---|
| `u`, `i` | `(B,)` |
| `p_u = user_emb(u)` | `(B,K)` |
| `f_u(p_u)` | `(B,K)` |
| `q_i = item_emb(i)` | `(B,K)` |
| `(f_u(p_u) * q_i).sum(dim=1)` | `(B,)` |
| prediction | `(B,1)` |

The output width of the tower must equal the item-embedding width `K`; otherwise the dot product is undefined.

## 2. The data from Warm-up 04

Each row of `X_ratings` is `[user_id, item_id]`; the matching entry of `y_ratings` is its rating. The full vocabulary is 5 users and 4 items, fixed before fitting and before every CV fold.

In [1]:
import numpy as np
import torch
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold
from skorch import NeuralNetRegressor
from torch import nn

torch.manual_seed(0)

n_users, n_items = 5, 4

X_ratings = np.array([
    [0, 0], [0, 1],
    [1, 0], [1, 1], [1, 2], [1, 3],
    [2, 0], [2, 2], [2, 3],
    [3, 1], [3, 2], [3, 3],
    [4, 0], [4, 1], [4, 2], [4, 3],
], dtype=np.float32)
y_ratings = np.array([
    5., 4.,
    1., 2., 2., 1.,
    4., 4., 5.,
    3., 4., 3.,
    2., 3., 3., 2.,
], dtype=np.float32).reshape(-1, 1)

mu = float(y_ratings.mean())

print('X shape:', X_ratings.shape)
print('y shape:', y_ratings.shape)
print('ID vocabulary:', n_users, 'users and', n_items, 'items')
print('observed ratings:', len(y_ratings))
print('global mean:', round(mu, 3))

X shape: (16, 2)
y shape: (16, 1)
ID vocabulary: 5 users and 4 items
observed ratings: 16
global mean: 3.0


## 3. Implement the user tower

Read `UserTowerNCF` in two passes:

1. **`__init__`** creates the two embedding tables, the two bias vectors, the global mean, and one small MLP `user_tower`.
2. **`forward`** looks up `p_u` and `q_i`, maps `p_u` through the tower, and takes the dot product.

The tower is `K → hidden → K`: the final width must match `q_i`.

In [2]:
class UserTowerNCF(nn.Module):
    def __init__(self, n_users, n_items, K=4, hidden=8, mu=0.0):
        super().__init__()
        torch.manual_seed(0)
        self.user_emb = nn.Embedding(n_users, K, max_norm=0.3)
        self.item_emb = nn.Embedding(n_items, K, max_norm=0.3)
        self.user_bias = nn.Parameter(torch.zeros(n_users, 1))
        self.item_bias = nn.Parameter(torch.zeros(n_items, 1))
        self.mu = nn.Parameter(torch.tensor([mu]))
        self.user_tower = nn.Sequential(
            nn.Linear(K, hidden), nn.ReLU(),
            nn.Linear(hidden, K),
        )

    def forward(self, x):
        u = x[:, 0].long()
        i = x[:, 1].long()
        pu = self.user_emb(u)
        fu = self.user_tower(pu)
        qi = self.item_emb(i)
        dot = (fu * qi).sum(dim=1, keepdim=True)
        return self.mu + self.user_bias[u] + self.item_bias[i] + dot


demo = UserTowerNCF(n_users, n_items, K=4, hidden=8, mu=mu)
print(demo)
print('learned numbers:', sum(p.numel() for p in demo.parameters()))
print('output shape:', tuple(demo(torch.as_tensor(X_ratings[:5])).shape))

UserTowerNCF(
  (user_emb): Embedding(5, 4, max_norm=0.3)
  (item_emb): Embedding(4, 4, max_norm=0.3)
  (user_tower): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=4, bias=True)
  )
)
learned numbers: 122
output shape: (5, 1)


**Checkpoint.** Why does `user_tower` end with `nn.Linear(hidden, K)` rather than `nn.Linear(hidden, 1)`? What happens to the model if you replace the whole tower with `nn.Identity()`?

## 4. Fit with `skorch`

`NeuralNetRegressor` supplies the `fit` / `predict` interface. We set `train_split=False` because `GridSearchCV` will create the validation folds later. This first fit uses all 16 observed ratings.

In [3]:
# Your turn: create the estimator and fit it.
# ncf = NeuralNetRegressor(...)
# ncf.fit(...)

### Solution

In [4]:
ncf = NeuralNetRegressor(
    module=UserTowerNCF,
    module__n_users=n_users,
    module__n_items=n_items,
    module__mu=mu,
    module__K=4,
    module__hidden=8,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    optimizer__lr=0.01,
    max_epochs=150,
    batch_size=4,
    iterator_train__shuffle=True,
    train_split=False,
    verbose=0,
)
ncf.fit(X_ratings, y_ratings)

train_pred = ncf.predict(X_ratings)
print('settings: K=4, hidden=8')
print('learned numbers:', sum(p.numel() for p in ncf.module_.parameters()))
print('final training loss:', round(float(ncf.history[-1, 'train_loss']), 4))
print('training RMSE:', round(root_mean_squared_error(y_ratings, train_pred), 4))
print('prediction shape:', train_pred.shape)

settings: K=4, hidden=8
learned numbers: 122
final training loss: 0.2128
training RMSE: 0.4579
prediction shape: (16, 1)


## 5. Predict a few pairs

`(0,0)` and `(3,1)` are observed training pairs. `(0,2)` and `(2,1)` are missing pairs with a known user and a known item. The predictions are not constrained to be integers.

In [5]:
X_new = np.array([
    [0, 0],  # observed pair
    [0, 2],  # missing pair, known user and item
    [2, 1],  # missing pair, known user and item
    [3, 1],  # observed pair
], dtype=np.float32)

print('predictions:', np.round(ncf.predict(X_new), 3))

predictions: [[4.111]
 [4.535]
 [4.303]
 [3.355]]


## 6. Choose $K$ and `hidden` with `GridSearchCV`

Compare `K = 2, 4` and `hidden = 4, 8` with shuffled four-fold cross-validation. Use `scoring='neg_root_mean_squared_error'`, `cv=KFold(n_splits=4, shuffle=True, random_state=0)`, and `refit=True`.

Before running: why are `P`, `Q`, `mu`, `a`, `b`, and the tower weights **not** in `param_grid`? What kind of quantity belongs there instead?

In [6]:
# Your turn: complete the GridSearchCV workflow.
# search = GridSearchCV(
#     estimator=NeuralNetRegressor(...),
#     param_grid={...},
#     scoring='...',
#     cv=...,
#     refit=True,
# )
# search.fit(X_ratings, y_ratings)

### Solution

In [7]:
search = GridSearchCV(
    estimator=NeuralNetRegressor(
        module=UserTowerNCF,
        module__n_users=n_users,
        module__n_items=n_items,
        module__mu=mu,
        criterion=nn.MSELoss,
        optimizer=torch.optim.SGD,
        optimizer__lr=0.01,
        max_epochs=150,
        batch_size=4,
        iterator_train__shuffle=True,
        train_split=False,
        verbose=0,
    ),
    param_grid={
        'module__K': [2, 4],
        'module__hidden': [4, 8],
    },
    scoring='neg_root_mean_squared_error',
    cv=KFold(n_splits=4, shuffle=True, random_state=0),
    refit=True,
)
search.fit(X_ratings, y_ratings)

rows = sorted(
    zip(
        search.cv_results_['param_module__K'],
        search.cv_results_['param_module__hidden'],
        search.cv_results_['mean_test_score'],
        search.cv_results_['std_test_score'],
    ),
    key=lambda row: -row[2],
)
for K, hidden, score, std in rows:
    print(f'K={int(K)}, hidden={int(hidden)}: '
          f'CV RMSE={-score:.3f} (std {std:.3f})')

print('selected settings:', search.best_params_)
print('best CV RMSE:', round(-search.best_score_, 3))
best_ncf = search.best_estimator_
print('refitted training RMSE:', round(root_mean_squared_error(
    y_ratings, best_ncf.predict(X_ratings)
), 3))
print('best-model predictions:', np.round(best_ncf.predict(X_new), 3))

K=2, hidden=8: CV RMSE=0.926 (std 0.236)
K=2, hidden=4: CV RMSE=0.936 (std 0.247)
K=4, hidden=8: CV RMSE=0.956 (std 0.247)
K=4, hidden=4: CV RMSE=0.960 (std 0.244)
selected settings: {'module__K': 2, 'module__hidden': 8}
best CV RMSE: 0.926
refitted training RMSE: 0.457
best-model predictions: [[4.142]
 [4.567]
 [4.297]
 [3.376]]


## 7. Read the result

`search.best_score_` is the **cross-validation** RMSE. The training RMSE of `search.best_estimator_` is a fit diagnostic on all 16 ratings. Neither one is a test-set score.

With only 16 ratings, four folds are small and the ranking can be noisy. The workflow still transfers directly to the Netflix data: fixed ID vocabulary → `fit` → choose with CV → refit.

## 8. The workflow map

| Step | Call | Question it answers |
|---|---|---|
| Fit one candidate | `ncf.fit(X, y)` | What does this tower learn? |
| Predict | `ncf.predict(X_new)` | What rating do we output? |
| Compare candidates | `search.fit(X, y)` | Which $K$ and `hidden` have the best CV RMSE? |
| Final model | `search.best_estimator_` | Which fitted model do we use after selection? |
| Report | `best_ncf.predict(X_new)` | What are the final predictions? |

The user tower changes the **interaction**, not the estimator interface.

## Exit ticket

1. What changes from biased MF to this NCF model?
2. Why must `f_u(p_u)` and `q_i` have the same width?
3. Which quantities are hyperparameters, and which are learned inside `fit`?
4. Why do we fix `n_users` and `n_items` before cross-validation?
5. What happens if the user tower is `nn.Identity()`?
6. Why is four-fold CV noisy on this 16-rating dataset?

<details>
<summary><b>After discussion</b></summary>

1. The dot product is replaced by $f_u(p_u)^\mathsf{T}q_i$: the user embedding goes through a small MLP before the interaction.
2. The dot product multiplies matching coordinates, so the tower output and the item embedding must both have width $K$.
3. $K$, `hidden`, the learning rate, the number of epochs, and the batch size are chosen before fitting. $\mu$, the biases, the embedding tables, and the tower weights are learned.
4. A CV fold may not contain every user or item. Fixed vocabulary sizes keep the embedding shapes valid, even when some rows are not updated in a fold.
5. The tower output becomes $p_u$, so the model reduces to biased MF.
6. Each fold has only four validation ratings, and some users or items may be absent from a training fold. The CV estimate is therefore unstable; the main notebook uses much more data.

</details>